# 3 — Direction FAB: Modern Spatio-Temporal Convolutions & Sequential Modelling

Direction 3 (`fab`) for the DOLOS Deception Detection project.

**Architecture Overview**:
- **Visual stream**: `swin3d_t` / `x3d_m` extracting spatio-temporal feature sequences $(T_v, 768)$ without collapsing time.
- **Acoustic stream**: Log-Mel spectrogram (3 channels: static $S$, velocity $\Delta$, acceleration $\Delta^2$) + `convnext_tiny` $(T_a, 768)$.
- **Temporal alignment & fusion**: 1D linear interpolation along time axis + projection to common space ($d=128$) + concatenation ($d=256$).
- **Sequential model**: 2-layer Bidirectional GRU (or Dilated TCN) capturing global temporal interview dynamics.
- **Aggregation**: Temporal Attention Pooling weighting emotional micro-leakage instants.

**Hard Constraints** (from `lie-detector/AGENTS.md` & `docs/evaluation.md`):
1. Do not touch `DATA/splits/` or `lib/`. Keep test set reserved; CV score is the mean over the 5 fold holdouts.
2. Optimize decision threshold on `val` (`best_threshold`), never default to 0.5 without tuning.
3. Save per-clip predictions to `preds/cv_fab.csv`.

## 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [22]:
# Clone or pull the repository to access `lib/` and the frozen manifests
import os
if not os.path.isdir("/content/DeepLearningBigData/.git"):
    !git clone -q -b CNN_GRU https://github.com/mattsays/DeepLearningBigData /content/DeepLearningBigData
%cd /content/DeepLearningBigData/lie-detector
!git checkout CNN_GRU
!git pull -q
import sys; sys.path.insert(0, '.')
# Autoreload (optional, commented out to avoid IPython 3.13 extension issues)
# %load_ext autoreload
# %autoreload 2


/content/DeepLearningBigData/lie-detector
Already on 'CNN_GRU'
Your branch is up to date with 'origin/CNN_GRU'.


In [23]:
from pathlib import Path

# Shared Drive folder. Must have a shortcut named 'dolos' under 'MyDrive'
DATA = Path("/content/drive/MyDrive/dolos")
print(DATA, "->", sorted(p.name for p in DATA.glob("*")))

/content/drive/MyDrive/dolos -> ['audio', 'cached_features', 'clips', 'dataset.csv', 'dolos_annotations.zip', 'splits']


In [24]:
from lib.dataset import cache_dir

DIRECTION = "fab"
ENCODER = "swin3d_convnext_3ch"         # Features extracted from Swin3D + ConvNeXt-3ch
CACHE = cache_dir(ENCODER, DATA)        # DATA/cached_features/<ENCODER>/<clip_id>.npy
SEED = 0
BATCH_SIZE = 16
EPOCHS = 35
PATIENCE = 7
FOLDS = 5
print(f"Direction: {DIRECTION} | Cache path: {CACHE}")

Direction: fab | Cache path: /content/drive/MyDrive/dolos/cached_features/swin3d_convnext_3ch


## 1 — The Frozen Splits

`Protocol.load(DATA / "splits")` loads the subject-independent protocol and verifies SHA-256 manifests.

In [25]:
from lib.dataset import Protocol, Clips

folds = Protocol.load(DATA / "splits")
clips = Clips.load(DATA)
print("Source fingerprint:", clips.describe_source())

train, val, holdout = folds.subject(1)
split_ids = [c for fold in range(1, FOLDS + 1)
             for part in folds.subject(fold) for c in part] + folds.test()
missing = clips.missing_from(split_ids)
print(f"{len(missing)} split clips are absent from dataset.csv")
if missing:
    print("  first missing:", missing[:3], "-> Splits and clips mismatch! Check dataset download.")

for name, ids in (("train", train), ("val", val), ("holdout", holdout)):
    print(f"{name:8}", clips.describe(ids))

Source fingerprint: dataset.csv sha256:8fa198b1fe71  1678 clips  1452 with media
0 split clips are absent from dataset.csv
train    {'clips': 1011, 'subjects': 94, 'deception': 56.1, 'pos_weight': 0.783, 'female': 17.0, 'majority_baseline': 56.1}
val      {'clips': 109, 'subjects': 16, 'deception': 47.7, 'pos_weight': 1.096, 'female': 28.4, 'majority_baseline': 52.3}
holdout  {'clips': 177, 'subjects': 27, 'deception': 48.0, 'pos_weight': 1.082, 'female': 28.2, 'majority_baseline': 52.0}


## 2 — Feature Loading & Offline Pre-Extraction

Loads pre-extracted sequence features `<clip_id>.npy` of shape `(T=8, 1536)` from `CACHE`.
- If features have not been extracted yet, run the extraction script cell below once.
- If the cache does not exist, the loader automatically falls back to deterministic `PseudoFeatures`
  with matching shape `(8, 1536)` for smoke-testing the training loop.

In [ ]:
# Run this cell ONCE on Colab with GPU runtime to extract all clips into Drive:
!python scripts/extract_features_fab.py --data-dir "$DATA" --encoder "$ENCODER"
print(f"Target cache folder: {CACHE}")
if CACHE.exists():
    cached_files = list(CACHE.glob('*.npy'))
    print(f"Found {len(cached_files)} cached feature files in {CACHE}")
else:
    print(f"Cache directory does not exist yet. Run extraction script or test with PseudoFeatures.")

Loaded 1678 rows from /content/drive/MyDrive/dolos/dataset.csv
Target feature cache: /content/drive/MyDrive/dolos/cached_features/swin3d_convnext_3ch
Clips to process: 1299 (skipped 379 already cached or missing media)
Loading extractor models on cuda...
Downloading: "https://download.pytorch.org/models/swin3d_t-7615ae03.pth" to /root/.cache/torch/hub/checkpoints/swin3d_t-7615ae03.pth
100% 122M/122M [00:00<00:00, 210MB/s] 

model.safetensors: downloading bytes:  30% 52.9M/178M [00:02<00:02, 45.1MB/s, 2.90MB/s  ]
model.safetensors: downloading bytes:  75% 134M/178M [00:03<00:00, 61.4MB/s, 11.0MB/s  ] 
model.safetensors: reconstructing file:  21% 37.3M/178M [00:03<00:15, 9.35MB/s, 3.29MB/s  ]
model.safetensors: downloading bytes:  81% 145M/178M [00:03<00:00, 60.6MB/s, 11.7MB/s  ]  ]
model.safetensors: downloading bytes:  87% 155M/178M [00:03<00:00, 67.2MB/s, 12.4MB/s  ]  ]
model.safetensors: downloading bytes:  92% 164M/178M [00:03<00:00, 70.1MB/s, 13.2MB/s  ] ] 
model.safetensors: downl

In [ ]:
import hashlib
import numpy as np
import torch
from torch.utils.data import Dataset
from lib.dataset import CachedFeatures, make_loader

DEBUG = not CACHE.exists()
if DEBUG:
    print(f"⚠️ No cache found at {CACHE} -> Falling back to pseudo-features for code testing only!")


class PseudoFeatures(Dataset):
    """Deterministic pseudo-features for pipeline smoke-testing with shape (8, 1536)."""
    def __init__(self, ids, clips, time_steps=8, feat_dim=1536):
        self.ids = clips.keep(ids)
        self.clips = clips
        self.time_steps = time_steps
        self.feat_dim = feat_dim

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        clip_id = self.ids[index]
        seed = int(hashlib.md5(clip_id.encode()).hexdigest()[:8], 16)
        rng = np.random.default_rng(seed)
        # Generate pseudo-feature tensor (Time, Dim) matching Swin3D (768) + ConvNeXt (768)
        x = torch.tensor(rng.normal(size=(self.time_steps, self.feat_dim)), dtype=torch.float32)
        return x, torch.tensor(self.clips[clip_id].y, dtype=torch.long)


def loader_for(ids, shuffle):
    dataset = PseudoFeatures(ids, clips) if DEBUG else CachedFeatures(ids, clips, CACHE)
    return make_loader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED)

## 3 — Direction 3 Model Architecture

Implements the Direction 3 sequential module supporting:
- **Bi-GRU** (Bidirectional Gated Recurrent Unit)
- **TCN** (Temporal Convolutional Network with dilated causal convolutions)
- **Temporal Attention Pooling** ($c = \sum \alpha_t h_t$) vs Mean Pooling
- MLP classifier with LayerNorm and Dropout.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class TemporalAttentionPooling(nn.Module):
    """Dynamically weights salient time steps (emotional micro-leakage)."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1, bias=False)
        )

    def forward(self, x):
        # x shape: (B, T, D)
        scores = self.attn(x)                       # (B, T, 1)
        weights = F.softmax(scores, dim=1)          # (B, T, 1)
        context = torch.sum(x * weights, dim=1)     # (B, D)
        return context, weights.squeeze(-1)


class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous() if self.chomp_size > 0 else x


class TemporalConvNet(nn.Module):
    """Dilated Temporal Convolutional Network (TCN)."""
    def __init__(self, in_dim, hidden_dim, num_levels=3, kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        for i in range(num_levels):
            dilation = 2 ** i
            in_ch = in_dim if i == 0 else hidden_dim
            padding = (kernel_size - 1) * dilation
            layers += [
                nn.Conv1d(in_ch, hidden_dim, kernel_size, padding=padding, dilation=dilation),
                Chomp1d(padding),
                nn.GELU(),
                nn.Dropout(dropout)
            ]
        self.net = nn.Sequential(*layers)
        self.downsample = nn.Conv1d(in_dim, hidden_dim, 1) if in_dim != hidden_dim else nn.Identity()

    def forward(self, x):
        # x: (B, T, D) -> Conv1d expects (B, D, T)
        xt = x.permute(0, 2, 1)
        res = self.downsample(xt)
        out = self.net(xt) + res
        return out.permute(0, 2, 1)                 # (B, T, hidden_dim)


class Direction3Model(nn.Module):
    """Direction 3: Sequential & Attention Classifier for Spatio-Temporal Features with Two-Stream LayerNorm."""
    def __init__(self, in_features, hidden_dim=128, seq_type="gru", pooling="attention", dropout=0.3):
        super().__init__()
        self.in_features = in_features
        self.seq_type = seq_type
        self.pooling = pooling
        
        # Two-Stream Modality Normalization & Balanced Projection
        if in_features >= 1536:
            self.v_proj = nn.Sequential(
                nn.LayerNorm(768),
                nn.Linear(768, hidden_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout / 2)
            )
            self.a_proj = nn.Sequential(
                nn.LayerNorm(768),
                nn.Linear(768, hidden_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout / 2)
            )
            self.fusion_proj = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout / 2)
            )
        else:
            self.v_proj = None
            self.a_proj = None
            self.single_proj = nn.Sequential(
                nn.LayerNorm(in_features),
                nn.Linear(in_features, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout / 2)
            )
        
        if seq_type == "gru":
            self.seq_module = nn.GRU(
                input_size=hidden_dim,
                hidden_size=hidden_dim,
                num_layers=2,
                batch_first=True,
                bidirectional=True,
                dropout=dropout if dropout > 0 else 0
            )
            seq_out_dim = hidden_dim * 2
        elif seq_type == "tcn":
            self.seq_module = TemporalConvNet(
                in_dim=hidden_dim,
                hidden_dim=hidden_dim,
                num_levels=3,
                kernel_size=3,
                dropout=dropout
            )
            seq_out_dim = hidden_dim
        else:
            self.seq_module = nn.Identity()
            seq_out_dim = hidden_dim

        if pooling == "attention":
            self.pool = TemporalAttentionPooling(seq_out_dim)
        else:
            self.pool = None

        self.classifier = nn.Sequential(
            nn.LayerNorm(seq_out_dim),
            nn.Dropout(dropout),
            nn.Linear(seq_out_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
        self.last_attn_weights = None

    def forward(self, x):
        # Handle both 2D (B, D) and 3D (B, T, D) inputs
        if x.dim() == 2:
            x = x.unsqueeze(1)
        
        if self.v_proj is not None and x.shape[-1] >= 1536:
            v = self.v_proj(x[..., :768])
            a = self.a_proj(x[..., 768:1536])
            h = self.fusion_proj(torch.cat([v, a], dim=-1))
        else:
            h = self.single_proj(x)
        
        if self.seq_type == "gru":
            seq_out, _ = self.seq_module(h)
        else:
            seq_out = self.seq_module(h)
            
        if self.pooling == "attention" and self.pool is not None:
            pooled, self.last_attn_weights = self.pool(seq_out)
        else:
            pooled = seq_out.mean(dim=1)
            
        logits = self.classifier(pooled)
        return logits.squeeze(-1)


first_x, _ = next(iter(loader_for(train, shuffle=False)))
in_dim = first_x.shape[-1]
model = Direction3Model(in_features=in_dim, hidden_dim=128, seq_type="gru", pooling="attention")
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Initialized Direction 3 Model | Input Dim: {in_dim} | Trainable Params: {n_trainable:,}")

## 4 — Training & Incremental Ablation Study on Fold 1

In accordance with `docs/evaluation.md`, all architectural trials and hyperparameter tuning are conducted
**strictly on Fold 1** (Train: Folds 2, 3, 4, 5 + captains; Val: 15% of train; Holdout: Fold 1).
- Compensates for class prior (~56.3% deception) using `pos_weight = clips.class_weights(train_ids)`.
- Dynamically tunes decision threshold on `val` (`best_threshold`).

In [ ]:
import copy
import torch
from lib.metrics import best_threshold, evaluate


def train_one_fold(
    fold=1,
    seq_type="gru",
    pooling="attention",
    modality="multimodal",      # 'multimodal' (full 1536), 'video_only' (0:768), 'audio_only' (768:1536)
    hidden_dim=128,
    lr=3e-4,
    dropout=0.3,
    verbose=True,
):
    train_ids, val_ids, hold_ids = folds.subject(fold)
    dl_train = loader_for(train_ids, shuffle=True)
    dl_val = loader_for(val_ids, shuffle=False)
    dl_hold = loader_for(hold_ids, shuffle=False)

    def slice_x(x_tensor):
        if modality == "video_only":
            return x_tensor[..., :768] if x_tensor.shape[-1] >= 768 else x_tensor
        elif modality == "audio_only":
            return x_tensor[..., 768:1536] if x_tensor.shape[-1] >= 1536 else x_tensor
        return x_tensor

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sample_x, _ = next(iter(dl_train))
    sample_x_sliced = slice_x(sample_x)
    actual_in_dim = sample_x_sliced.shape[-1]

    net = Direction3Model(
        in_features=actual_in_dim,
        hidden_dim=hidden_dim,
        seq_type=seq_type,
        pooling=pooling,
        dropout=dropout
    ).to(device)
    optimiser = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS, eta_min=1e-5)
    
    # Compensate for training split imbalance
    pos_weight = clips.class_weights(train_ids).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_loss, best_state, best_thr, waited = float("inf"), None, 0.5, 0
    for epoch in range(EPOCHS):
        net.train()
        for x, y in dl_train:
            x, y = slice_x(x).to(device), y.to(device)
            optimiser.zero_grad()
            logits = net(x)
            loss = loss_fn(logits, y.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
            optimiser.step()
        scheduler.step()

        net.eval()
        val_losses, probs, targets = [], [], []
        with torch.no_grad():
            for x, y in dl_val:
                x, y = slice_x(x).to(device), y.to(device)
                logits = net(x)
                val_losses.append(loss_fn(logits, y.float()).item())
                probs += torch.sigmoid(logits).cpu().tolist()
                targets += y.tolist()
                
        mean_val_loss = float(np.mean(val_losses))

        if mean_val_loss < best_val_loss:
            best_val_loss = mean_val_loss
            best_state = copy.deepcopy(net.state_dict())
            best_thr = best_threshold(targets, probs)
            waited = 0
        else:
            waited += 1
            if waited >= PATIENCE:
                break

    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    probs, targets = [], []
    with torch.no_grad():
        for x, y in dl_hold:
            x = slice_x(x).to(device)
            probs += torch.sigmoid(net(x)).cpu().tolist()
            targets += y.tolist()
            
    ids = clips.keep(hold_ids)
    metrics = evaluate(targets, probs, threshold=best_thr)
    if verbose:
        print(f"Fold {fold} [{seq_type.upper()}+{pooling.capitalize()} | {modality}]: "
              f"ACC {metrics['acc']*100:5.1f}%  F1 {metrics['f1']*100:5.1f}%  "
              f"AUC {metrics['auc']*100:5.1f}%  Balanced {metrics['balanced_acc']*100:5.1f}%  thr {best_thr:.2f}")
    return metrics, ids, targets, probs, net

### 4.1 Automated Ablation Study Matrix (Fold 1)

Runs the progressive comparison defined in `notes.html` and `notes.typ`:
1. **V3.0 (Baseline)**: Bi-GRU + Mean Pooling
2. **V3.1 (+ Attention)**: Bi-GRU + Temporal Attention Pooling
3. **V3.2a (Modality Ablation)**: Video Only + Attention
4. **V3.2b (Modality Ablation)**: Audio Only + Attention
5. **V3.3 (All-Convolutional)**: TCN (Dilated Temporal Conv1D) + Attention
6. **V3.3b (TCN + Mean)**: TCN + Mean Pooling

In [ ]:
import pandas as pd

ablation_configs = [
    {"name": "V3.0 (Baseline Bi-GRU + Mean Pool)", "seq": "gru", "pool": "mean", "mod": "multimodal"},
    {"name": "V3.1 (+ Temporal Attention Pooling)", "seq": "gru", "pool": "attention", "mod": "multimodal"},
    {"name": "V3.2a (Unimodal Video Only + Attn)", "seq": "gru", "pool": "attention", "mod": "video_only"},
    {"name": "V3.2b (Unimodal Audio Only + Attn)", "seq": "gru", "pool": "attention", "mod": "audio_only"},
    {"name": "V3.3 (All-Convolutional TCN + Attn)", "seq": "tcn", "pool": "attention", "mod": "multimodal"},
    {"name": "V3.3b (TCN + Mean Pooling)",         "seq": "tcn", "pool": "mean", "mod": "multimodal"},
]

print("=" * 75)
print("STARTING PHASE 3 ABLATION STUDY ON FOLD 1")
print("=" * 75)

ablation_records = []
trained_models = {}

for cfg in ablation_configs:
    m, ids, y_true, y_prob, net = train_one_fold(
        fold=1,
        seq_type=cfg["seq"],
        pooling=cfg["pool"],
        modality=cfg["mod"],
        verbose=False
    )
    trained_models[cfg["name"]] = net
    n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
    
    ablation_records.append({
        "Experiment": cfg["name"],
        "Modality": cfg["mod"],
        "Sequence": cfg["seq"].upper(),
        "Pooling": cfg["pool"].capitalize(),
        "Trainable Params": f"{n_params:,}",
        "Holdout ACC (%)": round(m["acc"] * 100, 1),
        "Holdout F1 (%)": round(m["f1"] * 100, 1),
        "Holdout AUC (%)": round(m["auc"] * 100, 1),
        "Balanced ACC (%)": round(m["balanced_acc"] * 100, 1),
        "Val Thr": round(m["threshold"], 2)
    })
    print(f"✓ Completed: {cfg['name']} -> Holdout AUC: {m['auc']*100:.1f}%, F1: {m['f1']*100:.1f}%")

df_ablation = pd.DataFrame(ablation_records)
print("\n" + "=" * 75)
print("ABLATION STUDY RESULTS (FOLD 1)")
print("=" * 75)
print(df_ablation.to_string(index=False))

### 4.2 Qualitative Analysis: Visualizing Temporal Attention Weights

Plots dynamic attention weights $\alpha_t$ across time for sample predictions from the
best attention model to verify if the network locates emotional micro-leakage instants.

In [ ]:
import matplotlib.pyplot as plt

# Use trained V3.1 attention model to inspect weights on Holdout Fold 1
target_model = trained_models.get("V3.1 (+ Temporal Attention Pooling)")

if target_model and target_model.pool is not None:
    target_model.eval()
    device = next(target_model.parameters()).device
    _, _, hold_ids = folds.subject(1)
    dl_hold = loader_for(hold_ids, shuffle=False)
    
    all_weights, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for x, y in dl_hold:
            x = x.to(device)
            logits = target_model(x)
            probs = torch.sigmoid(logits).cpu().tolist()
            w = target_model.last_attn_weights.cpu().numpy()  # (B, T)
            all_weights.append(w)
            all_targets.extend(y.tolist())
            all_probs.extend(probs)
            
    weights_matrix = np.concatenate(all_weights, axis=0)
    time_steps = np.arange(1, weights_matrix.shape[1] + 1)
    
    plt.figure(figsize=(10, 4.5))
    
    # Average weights for True Deceptions vs True Non-deceptions
    dec_mask = [t == 1 for t in all_targets]
    truth_mask = [t == 0 for t in all_targets]
    
    mean_dec_w = weights_matrix[dec_mask].mean(axis=0) if any(dec_mask) else weights_matrix.mean(axis=0)
    mean_truth_w = weights_matrix[truth_mask].mean(axis=0) if any(truth_mask) else weights_matrix.mean(axis=0)
    uniform_w = np.full_like(time_steps, 1.0 / len(time_steps), dtype=float)
    
    plt.plot(time_steps, mean_dec_w, marker='o', color='#dc2626', linewidth=2.5, label='Deception (Mean α_t)')
    plt.plot(time_steps, mean_truth_w, marker='s', color='#16a34a', linewidth=2.0, label='Truth (Mean α_t)')
    plt.plot(time_steps, uniform_w, linestyle='--', color='#94a3b8', linewidth=1.5, label='Uniform Baseline (1/T)')
    
    plt.title("Direction 3: Learned Temporal Attention Weights α_t across Clip Time Steps", fontsize=13, fontweight='bold')
    plt.xlabel("Temporal Step t (Interpolated Sequence Axis)", fontsize=11)
    plt.ylabel("Attention Weight α_t", fontsize=11)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(frameon=True, facecolor='white', framealpha=0.9)
    plt.tight_layout()
    plt.show()
    print("✓ Attention visualization completed.")
else:
    print("Run Section 4.1 first to initialize the attention model.")

## 5 — All 5 Folds Cross-Validation

Computes mean ± std over the 5 holdouts, evaluates pooled predictions against the
working set majority baseline (52.4%), and computes per-gender breakdowns.

In [ ]:
import numpy as np
import pandas as pd

results, pooled = [], []
for fold in range(1, FOLDS + 1):
    metrics, ids, targets, probs, _ = train_one_fold(fold, seq_type="gru", pooling="mean", modality="multimodal")
    results.append(metrics)
    pooled.append(pd.DataFrame({"clip_id": ids, "y_true": targets, "y_prob": probs}))

summary = pd.DataFrame(results)[["acc", "f1", "auc", "balanced_acc"]]
print("\n--- 5-Fold Cross Validation Results ---")
print(summary.round(3).to_string())
print(f"\nCV  ACC {summary.acc.mean()*100:.1f} ± {summary.acc.std()*100:.1f}  "
      f"F1 {summary.f1.mean()*100:.1f}  AUC {summary.auc.mean()*100:.1f}  "
      f"Balanced {summary.balanced_acc.mean()*100:.1f}")

In [ ]:
from lib.metrics import evaluate, majority_baseline

preds = pd.concat(pooled, ignore_index=True)
overall = evaluate(preds.y_true, preds.y_prob)
base = majority_baseline(preds.y_true)
print(f"Pooled ACC: {overall['acc']*100:.1f}%  Balanced: {overall['balanced_acc']*100:.1f}%  "
      f"| Majority baseline: {base*100:.1f}%")

# Save predictions for McNemar paired comparison
out = Path("preds")
out.mkdir(exist_ok=True)
preds_file = out / f"cv_{DIRECTION}.csv"
preds.to_csv(preds_file, index=False)
print(f"Saved out-of-fold predictions to {preds_file}")

In [ ]:
# Per-gender evaluation
meta = pd.read_csv(DATA / "dataset.csv", dtype=str).set_index("clip_id")
preds["gender"] = preds.clip_id.map(meta.gender)
print("Per-gender breakdown:")
for gender, group in preds.groupby("gender"):
    m = evaluate(group.y_true, group.y_prob)
    print(f"  {gender:7} n={m['n']:4d}  ACC {m['acc']*100:5.1f}%  Balanced {m['balanced_acc']*100:5.1f}%")

## 6 — Official Row for the Results Board

Prints the Markdown row ready to paste into `lie-detector/README.md`.

In [ ]:
from lib.metrics import format_row

cv_results = {
    "acc": summary.acc.mean(),
    "acc_std": summary.acc.std(),
    "f1": summary.f1.mean(),
    "auc": summary.auc.mean(),
    "balanced_acc": summary.balanced_acc.mean()
}
row = format_row(
    direction=DIRECTION,
    results=cv_results,
    owner="fab",
    method="Swin3D + ConvNeXt-3ch + Bi-GRU + Attn",
    params=n_trainable,
    majority=base,
    runtime="~2s/epoch"
)
print("\nPaste this row into lie-detector/README.md:\n")
print(row)